In [4]:
import os
import re
import json
import time
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pathlib import Path

In [5]:
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "data"

In [19]:
load_dotenv(override=True)

api_key = os.getenv("OPENROUTER_API_KEY")
if not api_key:
    raise ValueError("OPENROUTER_API_KEY not found. Please check your .env file.")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)

MODEL_NAME = "google/gemma-3-12b-it"

In [6]:
df_val = pd.read_csv(DATA_DIR / "MIPD" / "validation.csv")
print("Columns:", df_val.columns.tolist())

Columns: ['article', 'REFERENCE_ERROR', 'WHATABOUTISM', 'STRAWMAN', 'EMOTIONAL_CONTENT', 'CHERRY_PICKING', 'FALSE_CAUSE', 'MISLEADING_CLICKBAI', 'ANECDOTE', 'LEADING_QUESTIONS', 'EXAGGERATION', 'QUOTE_MINING']


In [30]:
article = 'article'
label_name = 'EXAGGERATION'
df_exaggeration = df_val[df_val[label_name] == 1].head(3)
df_no_exaggeration = df_val[df_val[label_name] == 0].head(2)

df = pd.concat([df_exaggeration, df_no_exaggeration]).reset_index(drop=True)

In [31]:
df[[article, label_name]]

,article,EXAGGERATION
0,"""Mieszkańcy Kolorado zagłosują w sprawie mordo...",True
1,"""W zeszłym roku Niemcy wydali na zasiłki dla i...",True
2,"""[TYLKO U NAS] Dr n. med. T. Wasilewski: Metod...",True
3,"""Szczepienia przeciw COVID-19: pora dnia moŋe ...",False
4,"""Koronawirus w Ameryce Łacińskiej. Chile wprow...",False


In [38]:
PROMPT_TEMPLATE = """You are an expert in media analysis and disinformation detection.
Analyze the following Polish article enclosed in <article> tags and determine whether it contains the manipulation technique: EXAGGERATION.

Definition: Exaggeration consists of overstating facts, overgeneralizing, or presenting phenomena in a disproportionately extreme manner without sufficient evidence.

<article>
{text}
</article>

Return your response EXCLUSIVELY as a single valid JSON object in the exact format below (no markdown blocks, no explanations, no additional text):
{{"decision": 1 or 0, "confidence": number_from_0_to_100}}
Where:
- "decision": integer 1 if EXAGGERATION is present, or 0 if it is absent.
- "confidence": integer from 0 to 100 indicating your confidence in this decision."""

In [39]:
def parse_json_response(raw_text):
    """Robustly extracts binary decision (0/1) and normalized confidence [0.0, 1.0]."""
    if not raw_text:
        return None, None
    
    # Layer 1: Standard JSON parsing on each {...} block individually
    matches = re.findall(r"\{[^{}]*\}", raw_text, re.DOTALL)
    for match in matches:
        try:
            data = json.loads(match)
            dec_raw = data.get("decision")
            conf_raw = data.get("confidence")
            if dec_raw is None or conf_raw is None:
                continue
            decision = int(dec_raw) if isinstance(dec_raw, bool) else int(str(dec_raw).strip())
            confidence = float(str(conf_raw).replace("%", "").strip())
            if decision in [0, 1] and 0 <= confidence <= 100:
                return decision, round(confidence / 100.0, 2)
        except Exception:
            continue

    # Layer 2: Regex fallback if the model outputs slightly malformed JSON syntax
    try:
        dec_match = re.search(r'["\']?decision["\']?\s*:\s*["\']?(1|0|true|false)["\']?', raw_text, re.IGNORECASE)
        conf_match = re.search(r'["\']?confidence["\']?\s*:\s*["\']?(\d+(?:\.\d+)?)%?["\']?', raw_text, re.IGNORECASE)
        if dec_match and conf_match:
            dec_val = dec_match.group(1).lower()
            decision = 1 if dec_val in ["1", "true"] else 0
            confidence = float(conf_match.group(1))
            if 0 <= confidence <= 100:
                return decision, round(confidence / 100.0, 2)
    except Exception:
        pass

    return None, None

def get_validated_annotation(prompt, temperature, top_p, seed, max_retries=5):
    """Queries the API and handles retries cleanly without leaving NaN values."""
    for attempt in range(max_retries):
        # If greedy (temp=0.0) fails on attempt 0, slightly increase temperature on retry to break deterministic loop
        effective_temp = temperature if (temperature > 0.0 or attempt == 0) else round(0.15 * attempt, 2)
        
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[{"role": "user", "content": prompt}],
                temperature=effective_temp,
                top_p=top_p,
                seed=seed + (attempt * 100)
            )
            content = response.choices[0].message.content if response.choices else None
            dec, conf = parse_json_response(content)
            
            if dec is not None and conf is not None:
                return int(dec), float(conf)
            else:
                snippet = repr(content[:120]) if content else "EMPTY_RESPONSE"
                print(f"  [Warning] Unparseable output on attempt {attempt + 1}/{max_retries}: {snippet}. Retrying...")
                time.sleep(1)
                
        except Exception as e:
            error_msg = str(e)
            if any(code in error_msg for code in ["401", "402", "404"]):
                raise RuntimeError(f"Fatal API configuration error: {error_msg}")
            print(f"  [Warning] Network error (attempt {attempt + 1}/{max_retries}): {e}. Retrying...")
            time.sleep(2)
            
    return None, None

In [ ]:
SEEDS = [42, 43, 44, 45, 46]
results = []

for index, row in df.iterrows():
    print(f"Processing item {index + 1}/{len(df)}...")
    text_sample = str(row[article])[:4000]
    gold_label = int(row[label_name])
    prompt = PROMPT_TEMPLATE.format(text=text_sample)
    
    # 1. Greedy reference output (temperature = 0.0)
    greedy_dec, greedy_conf = get_validated_annotation(
        prompt, temperature=0.0, top_p=1.0, seed=42
    )
    
    # 2. Five stochastic reruns (temperature = 0.7, top_p = 0.95)
    stochastic_decisions = []
    stochastic_confidences = []
    
    for seed in SEEDS:
        dec, conf = get_validated_annotation(
            prompt, temperature=0.7, top_p=0.95, seed=seed
        )
        if dec is not None:
            stochastic_decisions.append(int(dec))
            stochastic_confidences.append(float(conf))
        
    # 3. Compute confidence signals and correctness against human gold label
    n_valid = len(stochastic_decisions)
    if n_valid > 0:
        ones_count = stochastic_decisions.count(1)
        zeros_count = stochastic_decisions.count(0)
        
        majority_decision = 1 if ones_count > (n_valid / 2) else 0
        votes_for_majority = ones_count if majority_decision == 1 else zeros_count
        
        self_consistency = round(votes_for_majority / n_valid, 2)
        mean_verbalised_conf = round(sum(stochastic_confidences) / n_valid, 2)
        is_correct = (majority_decision == gold_label)
        
        if greedy_dec is None:
            greedy_dec = majority_decision
            greedy_conf = mean_verbalised_conf
    else:
        majority_decision = None
        self_consistency = None
        mean_verbalised_conf = None
        is_correct = None

    results.append({
        "item_id": index,
        "gold_label": gold_label,
        "greedy_decision": greedy_dec,
        "greedy_conf": greedy_conf,
        "stochastic_runs": stochastic_decisions,
        "stochastic_confs": stochastic_confidences,
        "majority_decision": majority_decision,
        "self_consistency": self_consistency,
        "mean_verbalised_conf": mean_verbalised_conf,
        "is_correct": is_correct
    })

results_df = pd.DataFrame(results)
display(results_df)
results_df.to_csv("data/results/poc_gemma3_results.csv", index=False)

Processing item 1/5...
Processing item 2/5...
Processing item 3/5...
Processing item 4/5...
Processing item 5/5...


,item_id,gold_label,greedy_decision,greedy_conf,stochastic_runs,stochastic_confs,majority_decision,self_consistency,mean_verbalised_conf,is_correct
0,0,1,1,0.85,"[1, 1, 1, 1, 1]","[0.85, 0.85, 0.9, 0.85, 0.85]",1,1.0,0.86,True
1,1,1,1,0.95,"[1, 1, 1, 1, 1]","[0.95, 0.95, 0.95, 0.95, 0.95]",1,1.0,0.95,True
2,2,1,1,0.85,"[1, 1, 1, 1, 1]","[0.85, 0.85, 0.85, 0.85, 0.85]",1,1.0,0.85,True
3,3,0,0,0.95,"[1, 0, 0, 0, 0]","[0.75, 0.95, 0.85, 0.95, 0.9]",0,0.8,0.88,True
4,4,0,1,0.85,"[1, 1, 1, 1, 1]","[0.85, 0.85, 0.85, 0.85, 0.85]",1,1.0,0.85,False
